In [1]:
import re
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

repo_id = "angelosbc/bertimbau-steam-sentiment"
tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModelForSequenceClassification.from_pretrained(repo_id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Léxicos de aspectos clássicos em reviews de jogos
ASPECTOS = {
    "Gráficos/Visual": [
        "grafico", "graficos", "visual", "textura", "cenario", "arte", "bonito", "feio"
    ],
    "Jogabilidade/Gameplay": [
        "gameplay", "jogabilidade", "mecanica", "controles", "combate", "movimentacao"
    ],
    "Preço/Custo-Benefício": [
        "preco", "preço", "caro", "barato", "valor", "promocao", "promoção", "reembolso", "centavo"
    ],
    "Desempenho/Técnico": [
        "fps", "otimizacao", "otimização", "lag", "crash", "bug", "travamento", "travar", "desempenho"
    ],
    "História/Áudio": [
        "historia", "história", "enredo", "narrativa", "trilha", "musica", "música", "dublagem", "som"
    ]
}

def classificar_trecho(texto):
    inputs = tokenizer(
        texto, padding=True, truncation=True, max_length=128, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    classe = "Recomenda" if probs[1] > probs[0] else "Não Recomenda"
    confianca = max(probs) * 100
    return classe, confianca

def analisar_aspectos(review):
    print("=" * 65)
    print(f"Review: \"{review}\"")
    print("=" * 65)

    # Divide o texto em orações usando pontuação e conjunções adversativas
    oracoes = re.split(r'[,.;!?]|\b(?:mas|porem|porém|contudo|entanto|todavia)\b', review, flags=re.IGNORECASE)
    oracoes = [o.strip() for o in oracoes if len(o.strip()) > 3]

    aspectos_encontrados = {asp: [] for asp in ASPECTOS}

    for oracao in oracoes:
        oracao_lower = oracao.lower()
        for asp, termos in ASPECTOS.items():
            if any(termo in oracao_lower for termo in termos):
                sentimento, score = classificar_trecho(oracao)
                aspectos_encontrados[asp].append((oracao, sentimento, score))

    # Exibição estruturada
    houve_aspecto = False
    for asp, ocorrencias in aspectos_encontrados.items():
        if ocorrencias:
            houve_aspecto = True
            print(f"\nAspecto: {asp}")
            for trecho, sent, score in ocorrencias:
                print(f"  - Trecho: \"{trecho}\"")
                print(f"    Polaridade: {sent} (Confiança: {score:.2f}%)")

    if not houve_aspecto:
        print("\nNenhum aspecto pré-definido foi identificado nos trechos.")
    print("-" * 65)

# Exemplo de teste com múltiplos aspectos
review_teste = "O visual e os gráficos são incríveis, a jogabilidade é fluida, mas o preço é muito caro e o jogo sofre com bugs e queda de fps."
analisar_aspectos(review_teste)

config.json:   0%|          | 0.00/954 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/678k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Review: "O visual e os gráficos são incríveis, a jogabilidade é fluida, mas o preço é muito caro e o jogo sofre com bugs e queda de fps."

Aspecto: Gráficos/Visual
  - Trecho: "O visual e os gráficos são incríveis"
    Polaridade: Recomenda (Confiança: 99.47%)

Aspecto: Jogabilidade/Gameplay
  - Trecho: "a jogabilidade é fluida"
    Polaridade: Recomenda (Confiança: 99.25%)

Aspecto: Preço/Custo-Benefício
  - Trecho: "o preço é muito caro e o jogo sofre com bugs e queda de fps"
    Polaridade: Não Recomenda (Confiança: 99.57%)

Aspecto: Desempenho/Técnico
  - Trecho: "o preço é muito caro e o jogo sofre com bugs e queda de fps"
    Polaridade: Não Recomenda (Confiança: 99.57%)
-----------------------------------------------------------------
